In [ ]:
import logging
import pathlib
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas
import torch
import tqdm
from matplotlib import gridspec
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import DataLoader

utils_path = pathlib.Path("../../utils/").resolve(strict=True)
sys.path.append(str(utils_path))

from model import TD_VAE, DBlock, Decoder, PreProcess
from prep_data import MNIST_Dataset
from rollout import rollout_func

/home/lippincm/miniforge3/envs/tdvae_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# set up logging
logger = logging.getLogger(__name__)
# make the log directory
pathlib.Path("../log").mkdir(exist_ok=True)
logging.basicConfig(filename="../log/training_log.log", level=logging.INFO)

In [3]:
# set path to the MNIST images
mnist_pickle_path = pathlib.Path("../../data/mnist/MNIST.pkl").resolve(strict=True)
# create the log directory if it does not exist
log_path = pathlib.Path("../log/").resolve()
log_path.mkdir(exist_ok=True)
log_file_path = pathlib.Path("../log/loginfo.txt").resolve()

In [4]:
with open(mnist_pickle_path, "rb") as file_handle:
    MNIST = pickle.load(file_handle)

# get the MNIST data keys
print(MNIST.keys())
MNIST["train_image"].shape

dict_keys(['train_image', 'train_label', 'test_image', 'test_label'])


(60000, 28, 28)

In [5]:
# set the batch size
batch_size = 512

# create the data class
# this class makes a rolling window of the data
data = MNIST_Dataset(
    MNIST["train_image"], MNIST["train_label"], binary=True, number_of_frames=20
)
# create the data loader
data_loader = DataLoader(data, batch_size=batch_size, shuffle=True)

In [6]:
## Hyperparameter optimization with Optuna
# Build a TD-VAE model
# dataset dependent constants

input_size = 784  # dataset dependent
processed_x_size = 784  # dataset dependent
# Set constants
time_constant_max = 16  # There are 20 frames total
time_jump_options = [1, 2, 3, 4]  # Jump up to 4 frames away

# hyper parameters
num_epochs = 1000
learning_rate = 0.0005
belief_state_size = 50  # hyperparameter
state_size = 8  # from original paper and hyperparameter
d_block_hidden_size = 50  # hyperparameter
decoder_hidden_size = 200  # hyperparameter

logger.info("Parameters set: ")
logger.info(f"num_epochs: {num_epochs}")
logger.info(f"learning_rate: {learning_rate}")
logger.info(f"belief_state_size: {belief_state_size}")
logger.info(f"state_size: {state_size}")
logger.info(f"d_block_hidden_size: {d_block_hidden_size}")
logger.info(f"decoder_hidden_size: {decoder_hidden_size}")
logger.info(f"batch_size: {batch_size}")
logger.info(f"input_size: {input_size}")
logger.info(f"processed_x_size: {processed_x_size}")
logger.info(f"time_constant_max: {time_constant_max}")
logger.info(f"time_jump_options: {time_jump_options}")

In [7]:
tdvae = TD_VAE(
    x_size=input_size,
    processed_x_size=processed_x_size,
    b_size=belief_state_size,
    z_size=state_size,
    d_block_hidden_size=d_block_hidden_size,
    decoder_hidden_size=decoder_hidden_size,
)
tdvae = tdvae.cuda()
# check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

In [ ]:
optimizer = optim.Adam(tdvae.parameters(), lr=learning_rate)
# make model save directory
model_save_dir = pathlib.Path("../output").resolve()
model_save_dir.mkdir(parents=True, exist_ok=True)

# Train the model

for epoch in range(num_epochs):
    epoch_loss = 0

    for batch, (idx, images) in enumerate(data_loader):
        batch_counter = 0
        batch_loss = 0
        images = images["image"].cuda()
        # Make a forward step of preprocessing and LSTM
        tdvae.forward(images)

        # Randomly sample a time step and jumpy step
        t_1 = np.random.choice(time_constant_max)
        t_2 = t_1 + np.random.choice(time_jump_options)

        # Calculate loss function based on two time points
        loss = tdvae.calculate_loss(t_1, t_2)
        if loss.isnan():
            print("loss is nan")
            pass
        elif loss.isinf():
            print("loss is inf")
            pass
        elif loss.item() == 0:
            print("loss is zero")
            pass
        elif loss.item() < 0:
            print("loss is negative")
            pass
        elif loss.item() > 0:
            batch_counter += 1
            batch_loss += loss.item()
            # must clear out stored gradient
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    epoch_loss += batch_loss / batch_counter
    logger.info(f"epoch: {epoch}, loss: {epoch_loss}")
    print("epoch: {:>4d}, loss: {:.4f}".format(epoch, epoch_loss))

    # save the model every 5 epochs and plot the jumpy reconstruction
    if (epoch + 1) % 50 == 0:
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": tdvae.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": loss,
            },
            pathlib.Path(model_save_dir / f"model_epoch_{epoch}.pt").resolve(),
        )
        plot = rollout_func(
            model_path=pathlib.Path(
                model_save_dir / f"model_epoch_{epoch}.pt"
            ).resolve(),
            input_size=input_size,
            processed_x_size=processed_x_size,
            belief_state_size=belief_state_size,
            state_size=state_size,
            d_block_hidden_size=d_block_hidden_size,
            decoder_hidden_size=decoder_hidden_size,
            mnist_pickle_path=mnist_pickle_path,
            epoch=epoch,
            batch_size=batch_size,
            num_frames=20,
            t1=16,
            t2=19,
        )
# save the final model
torch.save(
    {
        "epoch": epoch,
        "model_state_dict": tdvae.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": loss,
    },
    pathlib.Path(model_save_dir / "model_epoch_final.pt").resolve(),
)

epoch:    0, loss: 281.6396
epoch:    1, loss: 296.2421
epoch:    2, loss: 307.5884
epoch:    3, loss: 342.1243
epoch:    4, loss: 342.2811
epoch:    5, loss: 256.2048
epoch:    6, loss: 250.3797
epoch:    7, loss: 294.2546
epoch:    8, loss: 302.3032
epoch:    9, loss: 270.2003
epoch:   10, loss: 263.5232
epoch:   11, loss: 256.6388
epoch:   12, loss: 276.2729
epoch:   13, loss: 278.6781
epoch:   14, loss: 252.1894
epoch:   15, loss: 256.1988
epoch:   16, loss: 238.7088
epoch:   17, loss: 256.9006
epoch:   18, loss: 253.0807
epoch:   19, loss: 240.8827
epoch:   20, loss: 264.8697
epoch:   21, loss: 259.5075
epoch:   22, loss: 244.1861
epoch:   23, loss: 260.2484
epoch:   24, loss: 252.6010
epoch:   25, loss: 256.8584
epoch:   26, loss: 240.8523
epoch:   27, loss: 241.1918
epoch:   28, loss: 246.1728
epoch:   29, loss: 243.8675
epoch:   30, loss: 260.6923
epoch:   31, loss: 250.5435


In [ ]:
epoch = "final"
rollout_func(
    model_path=pathlib.Path(model_save_dir / f"model_epoch_{epoch}.pt").resolve(),
    input_size=input_size,
    processed_x_size=processed_x_size,
    belief_state_size=belief_state_size,
    state_size=state_size,
    d_block_hidden_size=d_block_hidden_size,
    decoder_hidden_size=decoder_hidden_size,
    mnist_pickle_path=mnist_pickle_path,
    epoch=epoch,
    batch_size=batch_size,
    num_frames=20,
    t1=16,
    t2=19,
)